# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a Croissant-formatted dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets defined in the metadata and their fields
from pprint import pprint
record_sets = [rs for rs in metadata.record_sets]

print(f"Available Record Sets (by @id):")
for record_set in record_sets:
    print(f"- {record_set['@id']} (name: {record_set['name']})")
    print("  Fields:")
    for field in record_set.get('fields', []):
        # Fields can be either dict or @id string; here get as dict
        if isinstance(field, dict):
            print(f"    - {field['@id']} (name: {field.get('name', '')}, type: {field.get('dataType', '')})")
        else:
            # It's just the @id
            print(f"    - {field}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Prepare to extract data from the main record set(s)
# (Choose the first non-empty record set for further steps)
main_record_set_id = None
for rs in record_sets:
    # The '@id' is the unique reference
    main_record_set_id = rs['@id']
    break  # Use the first one (if more, could allow a choice)

print(f"Using main record set: {main_record_set_id}")

# Load all records into a pandas DataFrame
records = list(dataset.records(record_set=main_record_set_id))
df = pd.DataFrame(records)
# Show the available columns (should be field @id's)
print('DataFrame columns:', df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, or grouping data by key attributes. This section demonstrates operations like removing outliers, transforming data distributions, or grouping data by field to prepare for further analysis.

In [ ]:
# Choose a numeric field by @id (find one with floating point or integer values)
sample_numeric_fields = [col for col in df.columns if df[col].dtype.kind in 'if' and (df[col].dropna().size > 0)]
print(f"Numeric Fields: {sample_numeric_fields}")

if len(sample_numeric_fields) > 0:
    numeric_field = sample_numeric_fields[0]
    print(f"Using numeric field: {numeric_field}")

    # Filter records where numeric_field > a threshold
    threshold = df[numeric_field].quantile(0.9)  # Use 90th percentile as flexible threshold
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records where {numeric_field} > {threshold} (90th percentile):")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a categorical field if there is one
    # Looking for a string/object column other than the numeric field
    cat_fields = [col for col in df.columns if df[col].dtype == 'object' and col != numeric_field]
    if cat_fields:
        group_field = cat_fields[0]
        print(f"Grouping by field: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index().sort_values(numeric_field, ascending=False)
        print("Mean of numeric field by group:")
        display(grouped_df.head())
else:
    print("No numeric fields found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(df) > 0 and len(sample_numeric_fields) > 0:
    # Histogram of the main numeric field
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # If group_field exists, boxplot
    if 'group_field' in locals():
        plt.figure(figsize=(12,6))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load, overview, filter, transform, and visualize a Croissant schema dataset with `mlcroissant`.

**Key steps covered:**
- Loading data and metadata via schema URL
- Exploring schema (record set and field @id)
- Extracting all records as a DataFrame
- Filtering and normalizing by field
- Grouping and basic data visualization

You can continue by exploring more fields, combining record sets, or extending EDA for downstream analysis.